# **Seismic Inversion FWI - FPN**

# **0. Install tensorflow probability**


In [1]:
import tensorflow as tf
print(tf.__version__)

2026-09-06 10:06:23.024736: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-06 10:06:23.088934: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-06 10:06:23.091190: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-09-06 10:06:23.091199: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore 

2.11.0


## **1. Python libraries**

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import math
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dropout

In [3]:
import tensorflow as tf

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


2026-09-06 10:06:24.598335: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-09-06 10:06:24.601524: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-09-06 10:06:24.601559: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2026-09-06 10:06:24.601582: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2026-09-06 10:06:24.601604: W tensorflow/c

In [4]:
!nvidia-smi

Sun Sep  6 10:06:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 Ti     Off |   00000000:01:00.0  On |                  N/A |
|  0%   43C    P8             25W /  310W |     460MiB /   8192MiB |     18%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **2. Paths**

### **2.1. Traces - InputData**

In [5]:
dirTraceTrain = "../Data/kimberlina_co2_train_data"
dirTraceVal = "../Data/kimberlina_co2_test_data"

### **2.2. Velocity - OutputData**

In [6]:
dirModelTrain = "../Model/kimberlina_co2_train_label"
dirModelVal = "../Model/kimberlina_co2_test_label"

### **2.3. Directory for saved trained models**

In [7]:
dirModelTrained="../ModelTrained/"
dirModel="../Model/"

### **2.4. Trained models directory**

In [8]:
dirImages = "../Images/"

# **3. Load list Paths**

### **3.1. Load list path Data Traces - Input**

In [9]:
TracePaths_train = sorted(
    os.path.join(dirTraceTrain, f)
    for f in os.listdir(dirTraceTrain)
    if f.endswith(".npz")
)


In [10]:
len(TracePaths_train)

15000

In [11]:
TracePaths_val = sorted(
    os.path.join(dirTraceVal, f)
    for f in os.listdir(dirTraceVal)
    if f.endswith(".npz")
)

In [12]:
len(TracePaths_val)

4430

### **3.2. Load Path Data Velocity - Output**

In [13]:
ModelPaths_train = sorted(
    os.path.join(dirModelTrain, f)
    for f in os.listdir(dirModelTrain)
    if f.endswith(".npz")
)

In [14]:
len(ModelPaths_train)

15000

In [15]:
ModelPaths_val = sorted(
    os.path.join(dirModelVal, f)
    for f in os.listdir(dirModelVal)
    if f.endswith(".npz")
)

In [16]:
len(ModelPaths_val)

4430

### **3.3.Splits**

In [17]:
# Semilla para reproducibilidad
random.seed(42)

nTest = 1000   # número de rutas para el test set, extraídas de train
nCal = 100    # número de rutas para el calibration set, extraídas de train

assert len(TracePaths_train) == len(ModelPaths_train), \
    "TracePaths_train y ModelPaths_train deben tener el mismo largo (pares input/output)"

# Generar índices mezclados para mantener la correspondencia input-output
indices = list(range(len(TracePaths_train)))
random.shuffle(indices)

test_idx = indices[:nTest]
cal_idx = indices[nTest:nTest + nCal]
train_idx = indices[nTest + nCal:]

# Aplicar los índices a ambas listas (trace y model) para no romper la correspondencia
TracePaths_test = [TracePaths_train[i] for i in test_idx]
ModelPaths_test = [ModelPaths_train[i] for i in test_idx]

TracePaths_cal = [TracePaths_train[i] for i in cal_idx]
ModelPaths_cal = [ModelPaths_train[i] for i in cal_idx]

TracePaths_train_final = [TracePaths_train[i] for i in train_idx]
ModelPaths_train_final = [ModelPaths_train[i] for i in train_idx]

print(f"Train final: {len(TracePaths_train_final)} pares")
print(f"Test: {len(TracePaths_test)} pares")
print(f"Calibration: {len(TracePaths_cal)} pares")
print(f"Validation (sin cambios): {len(TracePaths_val)} pares")

Train final: 13900 pares
Test: 1000 pares
Calibration: 100 pares
Validation (sin cambios): 4430 pares


### **3.4. Vmin and Vmax - Set Train**

In [18]:
vmin_global = np.inf
vmax_global = -np.inf

for path in ModelPaths_train:
    label = np.load(path)['label']
    vmin_global = min(vmin_global, label.min())
    vmax_global = max(vmax_global, label.max())

print(f"Min velocity: {vmin_global:.2f} m/s")
print(f"Max velocity: {vmax_global:.2f} m/s")

Min velocity: 951.82 m/s
Max velocity: 2546.00 m/s


# **4. Batch Creation**

In [19]:
import cv2

class SeismicSequence(tf.keras.utils.Sequence):
    def __init__(
        self,
        trace_paths,
        model_paths,
        batch_size=8,
        shuffle=True,
        target_size=(288,96)#(128, 128)
    ):
        self.trace_paths = trace_paths
        self.model_paths = model_paths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.target_size = target_size
        self.n_models = len(trace_paths)
        self.indices = np.arange(self.n_models)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(self.n_models / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_indices = self.indices[
            idx * self.batch_size : (idx + 1) * self.batch_size
        ]

        X_batch = []
        y_batch = []

        H, W = self.target_size

        for i in batch_indices:
            X = np.load(self.trace_paths[i])['data']   # (9, 1251, 101)
            y = np.load(self.model_paths[i])['label']  # (401, 141)

            y_resized = cv2.resize(y, (W, H), interpolation=cv2.INTER_LINEAR)  # (128, 128)

            X_batch.append(X)
            y_batch.append(y_resized)

        X_batch = np.asarray(X_batch, dtype=np.float32)   # (batch, 9, 1251, 101)
        y_batch = np.asarray(y_batch, dtype=np.float32)   # (batch, 128, 128)

        X_batch = X_batch.transpose(0, 2, 3, 1)           # (batch, 1251, 101, 9)
        y_batch = y_batch[:, :, :, np.newaxis]             # (batch, 128, 128, 1)

        v_min = vmin_global#np.min(y_batch, axis=(1, 2, 3), keepdims=True)
        v_max = vmax_global#np.max(y_batch, axis=(1, 2, 3), keepdims=True)
        
        y_batch = (y_batch - v_min) / (v_max - v_min)
        
        return X_batch, y_batch

In [20]:
nbatch=8
train_seq = SeismicSequence(
    trace_paths=TracePaths_train_final,
    model_paths=ModelPaths_train_final,
    batch_size=nbatch,
    shuffle=True
)

test_seq = SeismicSequence(
    trace_paths=TracePaths_test,
    model_paths=ModelPaths_test,
    batch_size=nbatch,
    shuffle=False
)

val_seq = SeismicSequence(
    trace_paths=TracePaths_val,
    model_paths=ModelPaths_val,
    batch_size=nbatch,
    shuffle=False
)

cal_seq = SeismicSequence(
    trace_paths=TracePaths_cal,
    model_paths=ModelPaths_cal,
    batch_size=nbatch,
    shuffle=False
)


In [21]:
print(f"Train batches: {len(train_seq)}")
print(f"Test batches: {len(test_seq)}")
print(f"Validation batches: {len(val_seq)}")
print(f"Calibration batches: {len(cal_seq)}")

Train batches: 1738
Test batches: 125
Validation batches: 554
Calibration batches: 13


In [22]:
print(len(TracePaths_train_final), len(ModelPaths_train_final))
print(len(TracePaths_val), len(ModelPaths_val))
print(len(TracePaths_test), len(ModelPaths_test))
print(len(TracePaths_cal), len(ModelPaths_cal))

13900 13900
4430 4430
1000 1000
100 100


In [23]:
for x,y in train_seq:
    break
x.shape,y.shape

((8, 1251, 101, 9), (8, 288, 96, 1))

# **5. Model Building**

In [24]:
import segmentation_models as sm

Segmentation Models: using `keras` framework.


In [25]:
BACKBONE = 'seresnet50'
#preprocess_input = sm.get_preprocessing(BACKBONE)

In [26]:
inp = layers.Input(shape=(1251, 101, 9))
x = layers.Conv2D(9, 1, activation="relu")(inp)
x = layers.MaxPooling2D(pool_size=(2, 1))(x)

x = layers.Conv2D(9, 1, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=(2, 1))(x)

2026-09-06 10:06:39.873277: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [27]:
x.shape

TensorShape([None, 312, 101, 9])

In [28]:
x = layers.Resizing(288, 96, interpolation="bilinear")(x)

In [29]:
x.shape

TensorShape([None, 288, 96, 9])

In [30]:
# define model
BB = sm.FPN(BACKBONE, input_shape=(288,96,9), decoder_filters=(2048,1024, 512, 256, 128, 64, 32), decoder_use_batchnorm=True, pyramid_use_batchnorm=True,encoder_weights=None, pyramid_dropout=0.1,activation="gelu", classes=1)(x)

/home/alejo/anaconda3/envs/AutoEncoder/lib/python3.7/site-packages/tensorflow_probability/python/layers/util.py:102: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use the `layer.add_weight()` method instead.
  trainable=trainable)
/home/alejo/anaconda3/envs/AutoEncoder/lib/python3.7/site-packages/tensorflow_probability/python/layers/util.py:112: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use the `layer.add_weight()` method instead.
  trainable=trainable)


In [31]:
model=Model(inp,BB)

In [32]:
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1251, 101, 9)]    0         
                                                                 
 conv2d (Conv2D)             (None, 1251, 101, 9)      90        
                                                                 
 max_pooling2d (MaxPooling2D  (None, 625, 101, 9)      0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 625, 101, 9)       90        
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 312, 101, 9)      0         
 2D)                                                             
                                                                 
 resizing (Resizing)         (None, 288, 96, 9)        0   

In [33]:
for layer in model.get_layer("model_1").layers:
    print(layer.name, layer.output_shape)

input [(None, 288, 96, 9)]
zero_padding2d (None, 294, 102, 9)
conv2d_2 (None, 144, 48, 64)
batch_normalization (None, 144, 48, 64)
activation (None, 144, 48, 64)
zero_padding2d_1 (None, 146, 50, 64)
max_pooling2d_2 (None, 72, 24, 64)
conv2d_3 (None, 72, 24, 64)
batch_normalization_1 (None, 72, 24, 64)
activation_1 (None, 72, 24, 64)
zero_padding2d_2 (None, 74, 26, 64)
conv2d_4 (None, 72, 24, 64)
batch_normalization_2 (None, 72, 24, 64)
activation_2 (None, 72, 24, 64)
conv2d_5 (None, 72, 24, 256)
batch_normalization_3 (None, 72, 24, 256)
global_average_pooling2d (None, 256)
lambda (None, 1, 1, 256)
conv2d_7 (None, 1, 1, 16)
activation_3 (None, 1, 1, 16)
conv2d_8 (None, 1, 1, 256)
activation_4 (None, 1, 1, 256)
conv2d_6 (None, 72, 24, 256)
multiply (None, 72, 24, 256)
batch_normalization_4 (None, 72, 24, 256)
add (None, 72, 24, 256)
activation_5 (None, 72, 24, 256)
conv2d_9 (None, 72, 24, 64)
batch_normalization_5 (None, 72, 24, 64)
activation_6 (None, 72, 24, 64)
zero_padding2d_3 (None,

In [34]:
def lossfunction(y_true, y_pred):
    return -y_pred.log_prob(y_true)

In [35]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
    #loss=tf.keras.losses.Huber(delta=1.0),
    loss=lossfunction,
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(),
        tf.keras.metrics.RootMeanSquaredError()
    ]
)

In [36]:
print(type(train_seq.batch_size), train_seq.batch_size)


<class 'int'> 8


# **6. Model Training**

In [37]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=50,
    restore_best_weights=True,
    verbose=1
)

In [39]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath=dirModelTrained+"0-Model-FPN-NegLik40b.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [40]:
callbacks = [
    lr_scheduler,
    early_stop,
    checkpoint
]

In [41]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-2)

In [ ]:
history = model.fit(train_seq,
                    validation_data=val_seq,
                    callbacks=callbacks,
                    epochs=40)

Epoch 1/40
1738/1738 [==============================] - ETA: 0s - loss: -22210.7676 - mean_absolute_error: 0.6147 - root_mean_squared_error: 6.1838     
Epoch 1: val_loss improved from inf to -24820.74023, saving model to ../ModelTrained/0-Model-FPN-NegLik40b.keras
1738/1738 [==============================] - 2244s 1s/step - loss: -22210.7676 - mean_absolute_error: 0.6147 - root_mean_squared_error: 6.1838 - val_loss: -24820.7402 - val_mean_absolute_error: 0.2989 - val_root_mean_squared_error: 3.3846 - lr: 0.0100
Epoch 2/40
1738/1738 [==============================] - ETA: 0s - loss: 98691592.0000 - mean_absolute_error: 3.4447 - root_mean_squared_error: 23.6664     
Epoch 2: val_loss did not improve from -24820.74023
1738/1738 [==============================] - 2212s 1s/step - loss: 98691592.0000 - mean_absolute_error: 3.4447 - root_mean_squared_error: 23.6664 - val_loss: -16495.4375 - val_mean_absolute_error: 3.3310 - val_root_mean_squared_error: 27.4442 - lr: 0.0100
Epoch 3/40
1738/17

# **7. Save model**

## **7.1. Save History**

In [ ]:
import pickle

with open(dirModelTrained+"history-NegLik40b.pkl", "wb") as f:
    pickle.dump(history.history, f)

In [ ]:
import json

# Convertimos cada lista de métricas a tipos nativos de Python
history_dict = {key: [float(i) for i in value] for key, value in history.history.items()}

with open(dirModelTrained + "history-NegLik40b.json", "w") as f:
    json.dump(history_dict, f)

## **7.2. Save Model**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
#from keras.saving import register_keras_serializable



In [ ]:
from tensorflow.keras.models import save_model

save_model(model, dirModelTrained + "weights-FPN-NegLik40b.keras")

In [ ]:
history.history